# 1. Load preprocessed dataset

In [2]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [3]:
import json
import pandas as pd

In [4]:
with open(r"data\sorted_processed_articles_corpus.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [5]:
df  = pd.DataFrame(data)

In [6]:
df

,article_index,url,processed_title_description,related_index
0,1,https://dantri.com.vn/thoi-su/ha-noi-duoc-trao...,"Hà_Nội trao quyền dự_án , giúp bộ_mặt thủ_đô ....","[162, 21244, 55]"
1,2,https://dantri.com.vn/thoi-su/cong-an-khong-ch...,“ Công_an bảo_vệ pháp_luật mà_còn đồng_hành dâ...,"[321, 15505, 18100]"
2,3,https://dantri.com.vn/thoi-su/tphcm-do-luong-h...,TPHCM đo_lường hạnh_phúc hài_lòng người_dân ? ...,"[394, 197, 13479]"
3,4,https://dantri.com.vn/thoi-su/xa-lu-gay-chet-n...,Xả lũ chết coi tội hình_sự . ( Dân_trí ) - Đ...,"[17670, 20175]"
4,5,https://dantri.com.vn/thoi-su/chuyen-tu-quan-l...,Chuyển “ quản_lý pháp_luật kiến_tạo thể_chế ...,[14475]
...,...,...,...,...
23177,25453,https://vnexpress.net/dot-quy-do-boc-tach-dong...,Đột_quỵ bóc tách động_mạch não . TP HCMAnh_Quâ...,[8794]
23178,25454,https://vnexpress.net/nu-sinh-an-mi-trong-gio-...,"Nữ_sinh mì học , cãi tay_đôi giảng_viên . Hà_N...",[10747]
23179,25455,https://dantri.com.vn/suc-khoe/ge-healthcare-h...,GE HealthCare hợp_tác Bệnh_viện Trung_ương Huế...,"[4932, 2463]"
23180,25456,https://vnexpress.net/ong-trump-lien-tuc-nham-...,Ông Trump liên_tục nhắm_mắt họp nội_các . Tổng...,[6384]


In [12]:
print(df.loc[18])

article_index                                                                 19
url                            https://dantri.com.vn/thoi-su/chinh-phu-de-xua...
processed_title_description    Chính_phủ đề_xuất chính_sách đặc_thù đường_sắt...
related_index                                                            [16870]
Name: 18, dtype: object


In [7]:
mismatch_rows = df[df['article_index'] != (df.index + 1)]

if len(mismatch_rows) > 0:
    first_mismatch = mismatch_rows.iloc[0]
    print("Dòng lệch đầu tiên:")
    print(first_mismatch)
    print(f"Index: {first_mismatch.name}, article_index: {first_mismatch['article_index']}")
else:
    print("Không có dòng nào lệch. Tất cả article_index = index + 1")

Dòng lệch đầu tiên:
article_index                                                                 21
url                            https://dantri.com.vn/thoi-su/hoc-va-lam-theo-...
processed_title_description    Học Bác , xây_dựng Sơn_La phát_triển . Gần 10 ...
related_index                                           [24944, 13065, 156, 326]
Name: 19, dtype: object
Index: 19, article_index: 21


In [21]:
print(df['processed_title_description'][0])

Hà_Nội trao quyền dự_án , giúp bộ_mặt thủ_đô . ( Dân_trí ) - Các dự_án , địa_bàn thủ_đô dự_án đầu_tư , cải_tạo công_trình nghẽn , cấp_bách ùn_tắc giao_thông , úng_ngập , ô_nhiễm môi_trường , trật_tự đô_thị ...


In [22]:
print(type(df['related_index'][0]))

<class 'list'>


In [23]:
print(type(df['article_index'][0]))

<class 'numpy.int64'>


# 2. Build Model

In [34]:
import numpy as np
import random
import torch
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from pathlib import Path
import os

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
    models,
    util
)
from sentence_transformers.trainer import SentenceTransformerTrainer
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [25]:
@dataclass
class TrainingConfig:
    base_model: str = "dangvantuan/vietnamese-embedding"

    # Training params
    batch_size: int = 32
    epochs: int = 5
    learning_rate: float = 2e-5
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01

    # Data params
    max_seq_length: int = 256
    num_hard_negatives: int = 3

    # Column mapping
    text_column: str = "processed_title_description"
    id_column: str = "article_index"
    relevant_column: str = "related_index"
    
    # Output
    output_dir: str = "./model"
    plots_dir: str = "./plots"
    
    # Evaluation
    eval_steps: int = 500

In [26]:
class MetricsTracker:    
    def __init__(self, config: TrainingConfig):
        self.config = config
        self.eval_steps: List[int] = []
        self.eval_results: List[Dict[str, float]] = []
        os.makedirs(config.plots_dir, exist_ok=True)
    
    def log(self, step: int, results: Dict[str, float]):
        self.eval_steps.append(step)
        self.eval_results.append(results)
    
    def get_metric_values(self, metric_key: str) -> List[float]:
        return [r.get(metric_key, 0) for r in self.eval_results]
    
    def get_best(self, metric_key: str) -> Tuple[int, float]:
        values = self.get_metric_values(metric_key)
        if not values:
            return 0, 0
        best_idx = np.argmax(values)
        return self.eval_steps[best_idx], values[best_idx]
    
    def save(self):
        data = {
            'eval_steps': self.eval_steps,
            'eval_results': self.eval_results,
        }
        with open(f"{self.config.plots_dir}/metrics.json", 'w') as f:
            json.dump(data, f, indent=2)

In [27]:
class TrackedIREvaluator:    
    def __init__(
        self, 
        ir_evaluator: evaluation.InformationRetrievalEvaluator, 
        tracker: MetricsTracker
    ):
        self.ir_evaluator = ir_evaluator
        self.tracker = tracker
    
    def __call__(self, model, output_path: str = None, epoch: int = -1, steps: int = -1):
        results = self.ir_evaluator(model, output_path) 
        current_step = steps if steps > 0 else epoch
        
        # Log metrics
        self.tracker.log(current_step, results)
        
        # Log summary
        mrr = results.get('ir_val_cosine_mrr@10', 0)
        recall = results.get('ir_val_cosine_recall@10', 0)
        logger.info(f"Step {current_step} - MRR@10: {mrr:.4f}, Recall@10: {recall:.4f}")
        
        return mrr

In [ ]:
class DataProcessor:
    def __init__(self, df: pd.DataFrame, config: TrainingConfig):
        self.config = config
        self.df = df.copy().explode(column=self.config.relevant_column)
        self.df = self.df.dropna(subset=[self.config.relevant_column])
        
        # Index to text mapping (full corpus)
        self.index_to_text = dict(
            zip(df[self.config.id_column], df[self.config.text_column])
        )
        self.all_indices = list(set(df[self.config.id_column]))
        
        self._split_data()
        
        logger.info(f"Data split:")
        logger.info(f" - Train queries: {len(self.queries_train)}, pairs: {len(self.train_df)}")
        logger.info(f" - Val queries: {len(self.queries_val)}, pairs: {len(self.val_df)}")
        logger.info(f" - Test queries: {len(self.queries_test)}, pairs: {len(self.test_df)}")
        logger.info(f" - Full corpus: {len(self.all_indices)} documents")
    
    def _split_data(self):
        self.queries_train_val, self.queries_test = train_test_split(
            self.all_indices, test_size=0.2, random_state=42
        )
        
        self.queries_train, self.queries_val = train_test_split(
            self.queries_train_val, test_size=0.1, random_state=42
        )
        
        self.train_val_df = self.df[self.df[self.config.id_column].isin(self.queries_train_val)]
        self.train_df = self.df[self.df[self.config.id_column].isin(self.queries_train)]
        self.val_df = self.df[self.df[self.config.id_column].isin(self.queries_val)]
        self.test_df = self.df[self.df[self.config.id_column].isin(self.queries_test)]
    
    def create_mnrl_examples(self, mode: str = "train") -> List[InputExample]:
        if mode == "train":
            df = self.train_df
        elif mode == "val":
            df = self.val_df
        else:
            df = self.test_df
            
        examples = []
        
        for _, row in df.iterrows():
            anchor = row[self.config.text_column]
            rel_idx = row[self.config.relevant_column]
            
            if rel_idx in self.index_to_text:
                positive = self.index_to_text[rel_idx]
                examples.append(InputExample(texts=[anchor, positive]))
        
        random.shuffle(examples)
        logger.info(f"Created {len(examples)} MNRL examples for {mode}")
        return examples
    
    def create_ir_evaluator(self, mode: str = "val") -> evaluation.InformationRetrievalEvaluator:
        """Tạo InformationRetrievalEvaluator chuẩn"""
        if mode == "val":
            query_df = self.val_df
        else:
            query_df = self.test_df
        
        # Queries: query_id -> query_text
        queries = {}
        # Corpus: doc_id -> doc_text (full corpus)
        corpus = {str(idx): self.index_to_text[idx] for idx in self.all_indices}
        # Relevant docs: query_id -> set of doc_ids
        relevant_docs = {}
        
        for _, row in query_df.iterrows():
            query_idx = row[self.config.id_column]
            rel_idx = row[self.config.relevant_column]
            
            query_id = str(query_idx)
            queries[query_id] = self.index_to_text[query_idx]
            
            if query_id not in relevant_docs:
                relevant_docs[query_id] = set()
            relevant_docs[query_id].add(str(rel_idx))
        
        logger.info(f"IR Evaluator ({mode}): {len(queries)} queries, {len(corpus)} corpus docs")
        
        return evaluation.InformationRetrievalEvaluator(
            queries=queries,
            corpus=corpus,
            relevant_docs=relevant_docs,
            name=f"ir_{mode}",
            mrr_at_k=[10],
            map_at_k=[10],
            ndcg_at_k=[10],
            show_progress_bar=True
        )


In [29]:
class TrainingVisualizer:
    def __init__(self, config: TrainingConfig):
        self.config = config
        os.makedirs(config.plots_dir, exist_ok=True)
    
    def plot_eval_metrics(self, tracker: MetricsTracker, save: bool = True):
        """
        Plot evaluation metrics
        
        Metrics: MRR@10, Recall@10, Precision@10, Accuracy@10, NDCG@10, MAP@100
        """
        if not tracker.eval_results:
            logger.warning("No eval results to plot")
            return
        
        metrics_to_plot = [
            ('ir_val_cosine_mrr@10', 'MRR@10'),
            ('ir_val_cosine_recall@10', 'Recall@10'),
            ('ir_val_cosine_precision@10', 'Precision@10'),
            ('ir_val_cosine_accuracy@10', 'Accuracy@10'),
            ('ir_val_cosine_ndcg@10', 'NDCG@10'),
            ('ir_val_cosine_map@100', 'MAP@100'),
        ]
        
        available_metrics = []
        for key, name in metrics_to_plot:
            values = tracker.get_metric_values(key)
            if any(v > 0 for v in values):
                available_metrics.append((key, name, values))
        
        if not available_metrics:
            logger.warning("No metrics data available")
            return
        
        steps = tracker.eval_steps
        
        n_metrics = len(available_metrics)
        n_cols = 2
        n_rows = (n_metrics + 1) // 2
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5 * n_rows))
        axes = axes.flatten() if n_metrics > 1 else [axes]
        
        colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#F44336', '#00BCD4']
        
        for i, (key, name, values) in enumerate(available_metrics):
            ax = axes[i]
            
            ax.plot(steps, values, color=colors[i % len(colors)], 
                   linewidth=2, marker='o', markersize=5, label=name)
            
            # Mark best point
            best_idx = np.argmax(values)
            ax.scatter([steps[best_idx]], [values[best_idx]], 
                      color='red', s=150, zorder=5, marker='*', 
                      label=f'Best: {values[best_idx]:.4f}')
            
            ax.set_xlabel('Training Steps')
            ax.set_ylabel('Score')
            ax.set_title(name)
            ax.legend(loc='lower right')
            ax.grid(True, alpha=0.3)
            ax.set_ylim(0, 1)
        
        # Hide empty subplots
        for i in range(len(available_metrics), len(axes)):
            axes[i].set_visible(False)
        
        plt.tight_layout()
        if save:
            plt.savefig(f"{self.config.plots_dir}/eval_metrics.png", dpi=150)
        plt.show()
        
        # Log best scores
        logger.info("Best scores:")
        for key, name, values in available_metrics:
            best_step, best_val = tracker.get_best(key)
            logger.info(f"  {name}: {best_val:.4f} at step {best_step}")
    
    def plot_all_metrics_combined(self, tracker: MetricsTracker, save: bool = True):
        if not tracker.eval_results:
            return
        
        metrics_to_plot = [
            ('ir_val_cosine_mrr@10', 'MRR@10'),
            ('ir_val_cosine_recall@10', 'Recall@10'),
            ('ir_val_cosine_precision@10', 'Precision@10'),
            ('ir_val_cosine_accuracy@10', 'Accuracy@10'),
        ]
        
        steps = tracker.eval_steps
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0']
        
        for i, (key, name) in enumerate(metrics_to_plot):
            values = tracker.get_metric_values(key)
            if any(v > 0 for v in values):
                ax.plot(steps, values, color=colors[i % len(colors)], 
                       linewidth=2, marker='o', markersize=4, label=name)
        
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Score')
        ax.set_title('Evaluation Metrics Over Training')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)
        
        plt.tight_layout()
        if save:
            plt.savefig(f"{self.config.plots_dir}/eval_metrics_combined.png", dpi=150)
        plt.show()
    
    def plot_similarity_distribution(
        self, 
        model: SentenceTransformer, 
        data_processor: DataProcessor,
        mode: str = "test",
        num_samples: int = 500,
        save: bool = True
    ):
        """Plot similarity distribution"""
        if mode == "test":
            df = data_processor.test_df
        else:
            df = data_processor.val_df
        
        df_sample = df.sample(min(num_samples, len(df)))
        
        relevant_sims = []
        non_relevant_sims = []
        
        for _, row in df_sample.iterrows():
            anchor_idx = row[data_processor.config.id_column]
            rel_idx = row[data_processor.config.relevant_column]
            
            if anchor_idx not in data_processor.index_to_text:
                continue
            if rel_idx not in data_processor.index_to_text:
                continue
            
            anchor_text = data_processor.index_to_text[anchor_idx]
            rel_text = data_processor.index_to_text[rel_idx]
            
            # Relevant similarity
            embs = model.encode([anchor_text, rel_text], normalize_embeddings=True)
            relevant_sims.append(float(np.dot(embs[0], embs[1])))
            
            # Random negative
            neg_pool = list(set(data_processor.all_indices) - {anchor_idx, rel_idx})
            if neg_pool:
                neg_idx = random.choice(neg_pool)
                neg_text = data_processor.index_to_text[neg_idx]
                neg_embs = model.encode([anchor_text, neg_text], normalize_embeddings=True)
                non_relevant_sims.append(float(np.dot(neg_embs[0], neg_embs[1])))
        
        if not relevant_sims:
            logger.warning("No similarity data to plot")
            return
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.hist(non_relevant_sims, bins=50, alpha=0.6, color='#F44336', 
                label=f'Non-relevant (n={len(non_relevant_sims)})', density=True)
        ax.hist(relevant_sims, bins=50, alpha=0.6, color='#4CAF50',
                label=f'Relevant (n={len(relevant_sims)})', density=True)
        
        separation = np.mean(relevant_sims) - np.mean(non_relevant_sims)
        ax.axvline(np.mean(relevant_sims), color='#2E7D32', linestyle='--', linewidth=2)
        ax.axvline(np.mean(non_relevant_sims), color='#C62828', linestyle='--', linewidth=2)
        
        ax.set_title(f'Similarity Distribution ({mode} set) - Separation: {separation:.3f}')
        ax.set_xlabel('Cosine Similarity')
        ax.set_ylabel('Density')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        if save:
            plt.savefig(f"{self.config.plots_dir}/similarity_distribution_{mode}.png", dpi=150)
        plt.show()
        
        logger.info(f"Separation ({mode}): {separation:.4f}")

    def plot_all(self, tracker: MetricsTracker, model: SentenceTransformer, data_processor: DataProcessor):
        # Plot eval metrics
        self.plot_eval_metrics(tracker)
        self.plot_all_metrics_combined(tracker)
        
        # Plot similarity distribution
        self.plot_similarity_distribution(model, data_processor, mode="test")


In [35]:
class EmbeddingTrainer:
    def __init__(self, config: TrainingConfig, data_processor: DataProcessor):
        self.config = config
        self.data_processor = data_processor
        self.model = None
        self.tracker = MetricsTracker(config)
        self.visualizer = TrainingVisualizer(config)
        
    def load_model(self):
        logger.info(f"Loading model: {self.config.base_model}")
        self.model = SentenceTransformer(self.config.base_model)
        self.model.max_seq_length = self.config.max_seq_length
        return self.model
    
    def train_mnrl(self) -> SentenceTransformer:
        """Train với MNRL"""
        if self.model is None:
            self.load_model()
        
        train_examples = self.data_processor.create_mnrl_examples(mode="train")
        
        if len(train_examples) == 0:
            raise ValueError("No training examples!")
        
        train_dataloader = DataLoader(
            train_examples,
            shuffle=True,
            batch_size=self.config.batch_size
        )
        
        train_loss = losses.MultipleNegativesRankingLoss(
            model=self.model,
            scale=20.0,
            similarity_fct=util.cos_sim
        )
        
        # Evaluator với tracking
        ir_evaluator = self.data_processor.create_ir_evaluator(mode="val")
        evaluator = TrackedIREvaluator(ir_evaluator, self.tracker)
        
        total_steps = len(train_dataloader) * self.config.epochs
        warmup_steps = int(total_steps * self.config.warmup_ratio)
        
        logger.info(f"Training MNRL: {len(train_examples)} pairs, {total_steps} steps")
        
        # Train
        self.model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=self.config.epochs,
            warmup_steps=warmup_steps,
            evaluator=evaluator,
            evaluation_steps=self.config.eval_steps,
            output_path=self.config.output_dir,
            save_best_model=True,
            optimizer_params={'lr': self.config.learning_rate},
            weight_decay=self.config.weight_decay,
            show_progress_bar=True,
            use_amp=True,
        )
        
        return self.model

    def train_triplet_hard_negatives(self) -> SentenceTransformer:
        """Train với Triplet Loss + Hard Negatives"""
        if self.model is None:
            self.load_model()
        
        train_examples = self.data_processor.create_triplet_examples(
            model=self.model, mode="train"
        )
        
        if len(train_examples) == 0:
            raise ValueError("No training triplets!")
        
        train_dataloader = DataLoader(
            train_examples, 
            shuffle=True, 
            batch_size=self.config.batch_size
        )
        
        train_loss = losses.TripletLoss(
            model=self.model,
            distance_metric=losses.TripletDistanceMetric.COSINE,
            triplet_margin=0.5
        )
        
        # Evaluator với tracking
        ir_evaluator = self.data_processor.create_ir_evaluator(mode="val")
        evaluator = TrackedIREvaluator(ir_evaluator, self.tracker)
        
        total_steps = len(train_dataloader) * self.config.epochs
        warmup_steps = int(total_steps * self.config.warmup_ratio)
        
        logger.info(f"Training Triplet (Hard Neg): {len(train_examples)} triplets, {total_steps} steps")
        
        # Train
        self.model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=self.config.epochs,
            warmup_steps=warmup_steps,
            evaluator=evaluator,
            evaluation_steps=self.config.eval_steps,
            output_path=self.config.output_dir,
            save_best_model=True,
            optimizer_params={'lr': self.config.learning_rate},
            weight_decay=self.config.weight_decay,
            show_progress_bar=True,
            use_amp=True,
        )
        
        return self.model
    
    def evaluate_on_test(self) -> Dict[str, float]:
        logger.info("\n" + "="*50)
        logger.info("Final Evaluation on TEST SET")
        logger.info("="*50)
        
        test_evaluator = self.data_processor.create_ir_evaluator(mode="test")
        
        results = test_evaluator(self.model, output_path=self.config.output_dir)
        
        # Extract metrics
        test_metrics = {
            'mrr@10': results.get('ir_test_cosine_mrr@10', 0),
            'recall@10': results.get('ir_test_cosine_recall@10', 0),
            'precision@10': results.get('ir_test_cosine_precision@10', 0),
            'accuracy@10': results.get('ir_test_cosine_accuracy@10', 0),
            'ndcg@10': results.get('ir_test_cosine_ndcg@10', 0),
            'map@100': results.get('ir_test_cosine_map@100', 0),
        }
        
        logger.info(f"  MRR@10:       {test_metrics['mrr@10']:.4f}")
        logger.info(f"  Recall@10:    {test_metrics['recall@10']:.4f}")
        logger.info(f"  Precision@10: {test_metrics['precision@10']:.4f}")
        logger.info(f"  Accuracy@10:  {test_metrics['accuracy@10']:.4f}")
        logger.info(f"  NDCG@10:      {test_metrics['ndcg@10']:.4f}")
        logger.info(f"  MAP@100:      {test_metrics['map@100']:.4f}")
        
        return test_metrics


In [ ]:
def train_model(
    df: pd.DataFrame,
    config: TrainingConfig = None,
    method: str = "mnrl",
    generate_plots: bool = True
) -> Tuple[SentenceTransformer, Dict[str, float]]:

    if config is None:
        config = TrainingConfig()
    
    # 1. Process data
    logger.info("Processing data...")
    data_processor = DataProcessor(df, config)
    
    # 2. Train
    trainer = EmbeddingTrainer(config, data_processor)
    
    if method == "mnrl":
        model = trainer.train_mnrl()
    elif method == "triplet":
        model = trainer.train_triplet_hard_negatives()
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # 3. Load best model
    logger.info(f"Loading best model from {config.output_dir}")
    model = SentenceTransformer(config.output_dir)
    trainer.model = model
    
    # 4. Evaluate on test
    test_metrics = trainer.evaluate_on_test()
    
    logger.info("\n" + "="*50)
    logger.info("TEST RESULTS:")
    logger.info(f"  MRR@10:       {test_metrics.get('mrr@10', 0):.4f}")
    logger.info(f"  Recall@10:    {test_metrics.get('recall@10', 0):.4f}")
    logger.info(f"  Precision@10: {test_metrics.get('precision@10', 0):.4f}")
    logger.info(f"  Accuracy@10:  {test_metrics.get('accuracy@10', 0):.4f}")
    logger.info(f"  NDCG@10:      {test_metrics.get('ndcg@10', 0):.4f}")
    logger.info(f"  MAP@100:      {test_metrics.get('map@100', 0):.4f}")
    logger.info("="*50)
    
    # 5. Plots
    if generate_plots:
        trainer.visualizer.plot_all(trainer.tracker, model, data_processor)
    
    # Save metrics
    trainer.tracker.save()
    
    return model, test_metrics

In [ ]:
if __name__ == "__main__":

    config = TrainingConfig(
        base_model="dangvantuan/vietnamese-embedding",
        batch_size=16,
        epochs=3,
        eval_steps=50,
        output_dir="./model",
        plots_dir="./plots",
    )
    
    # MNRL
    model, test_metrics = train_model(
        df=df,
        config=config,
        method="mnrl",
        generate_plots=True
    )
    
    # Hoặc train với Triplet + Hard Negatives
    # model, test_metrics = train_model(
    #     df=df,
    #     config=config,
    #     method="triplet",
    #     generate_plots=True
    # )
    
    print("\nDone!")

# 3. Qdrant

In [17]:
from qdrant_client import QdrantClient
client = QdrantClient(url = "http://localhost:6333")

In [18]:
from qdrant_client.models import Distance, VectorParams

client.create_collection(collection_name = "Article_Retrieval", 
                         vectors_config=VectorParams(size = 4, distance=Distance.COSINE))

ResponseHandlingException: [WinError 10061] No connection could be made because the target machine actively refused it

In [ ]:
from qdrant_client.models import PointStruct

operation_info = client.upsert(
    collection_name = "Article Retrieval",
    wait = True,
    points=[
        PointStruct(id=1, vector=[0.05, 0.61, 0.76, 0.74], payload={"city": "Berlin"}),
        PointStruct(id=2, vector=[0.19, 0.81, 0.75, 0.11], payload={"city": "London"}),
        PointStruct(id=3, vector=[0.36, 0.55, 0.47, 0.94], payload={"city": "Moscow"}),
        PointStruct(id=4, vector=[0.18, 0.01, 0.85, 0.80], payload={"city": "New York"}),
        PointStruct(id=5, vector=[0.24, 0.18, 0.22, 0.44], payload={"city": "Beijing"}),
        PointStruct(id=6, vector=[0.35, 0.08, 0.11, 0.44], payload={"city": "Mumbai"}),
    ],
)

In [ ]:
print(operation_info)

operation_id=1 status=<UpdateStatus.COMPLETED: 'completed'>


In [ ]:
search_result = client.query_points(
    collection_name="Article Retrieval",
    query = [0.2, 0.1, 0.9, 0.7],
    with_payload=False,
    limit=3
).points

In [ ]:
print(search_result)

[ScoredPoint(id=4, version=1, score=0.99248314, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=1, score=0.89463294, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=5, version=1, score=0.8543979, payload=None, vector=None, shard_key=None, order_value=None)]


In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

search_result = client.query_points(
    collection_name="Article Retrieval",
    query=[0.2, 0.1, 0.9, 0.7],
    query_filter=Filter(
        must=[FieldCondition(key="city", match=MatchValue(value="London"))]
    ),
    with_payload=True,
    limit=3,
    with_vectors = True
).points

print(search_result)

[ScoredPoint(id=2, version=1, score=0.66603535, payload={'city': 'London'}, vector=[0.16881056, 0.71966606, 0.66635746, 0.097732425], shard_key=None, order_value=None)]
